In [1]:
import pandas as pd
import numpy as np
df=pd.read_csv(r"C:\Users\rrama\OneDrive\Desktop\Python code\ecommerce_orders_dataset.csv")
print(df)

      order_id  order_date  customer_id       city product_category  \
0            1  2023-03-13          189    Kolkata      Electronics   
1            2  2023-03-29          100      Delhi         Clothing   
2            3  2023-10-04          161  Hyderabad      Electronics   
3            4  2024-04-19           59      Delhi      Electronics   
4            5  2024-07-20          244      Delhi      Electronics   
...        ...         ...          ...        ...              ...   
1195      1196  2024-07-13          191    Kolkata         Clothing   
1196      1197  2024-02-05          296  Hyderabad      Electronics   
1197      1198  2024-09-24          226     Mumbai      Electronics   
1198      1199  2023-08-04           43    Chennai             Home   
1199      1200  2024-01-25           22    Chennai           Sports   

     product_name  quantity  price  discount_percent    payment_method  \
0          Laptop         4   1160                10        Debit Card   

In [2]:
#Calculate revenue contribution percentage of each product category within every city.
df['revenue']=df['price']*df['quantity']
city_product_revenue=df.groupby(['city','product_category'])['revenue'].sum()
city_revenue=df.groupby('city')['revenue'].sum()
revenue_percentage=(city_product_revenue/city_revenue)*100
result=revenue_percentage.reset_index(name='revenue_percentage')
print(result)

         city product_category  revenue_percentage
0   Bangalore         Clothing           25.533999
1   Bangalore      Electronics           32.608131
2   Bangalore             Home           20.849081
3   Bangalore           Sports           21.008790
4     Chennai         Clothing           26.527625
5     Chennai      Electronics           20.451124
6     Chennai             Home           28.534444
7     Chennai           Sports           24.486807
8       Delhi         Clothing           19.068058
9       Delhi      Electronics           33.839758
10      Delhi             Home           25.580663
11      Delhi           Sports           21.511521
12  Hyderabad         Clothing           22.384784
13  Hyderabad      Electronics           25.541952
14  Hyderabad             Home           20.916462
15  Hyderabad           Sports           31.156802
16    Kolkata         Clothing           25.837582
17    Kolkata      Electronics           29.837704
18    Kolkata             Home 

In [3]:
#Find top 2 customers in each city based on total revenue generated.
df['revenue']=df['price']*df['quantity']
each_city=df.groupby(['city','customer_id'])['revenue'].sum().reset_index()
top_2_customer=each_city.sort_values(['city','revenue'],ascending=[True,False])\
    .groupby('city')\
    .head(2)
print(top_2_customer)


          city  customer_id  revenue
102  Bangalore          237   471731
12   Bangalore           28   322733
209    Chennai          192   459100
180    Chennai          128   443389
382      Delhi          298   423120
360      Delhi          239   350571
491  Hyderabad          245   430136
466  Hyderabad          187   343351
519    Kolkata            8   369191
623    Kolkata          258   365918
661     Mumbai           42   315913
752     Mumbai          253   315554
795       Pune           47   367290
837       Pune          145   321369


In [4]:
#Determine which product category has the highest return rate relative to its total orders.
returned_rate=df.groupby('product_category').agg(total_orders=('order_id','count'),returned_orders=('returned','sum'))
returned_rate['returned_rate']=returned_rate['returned_orders']/returned_rate['total_orders']
highest_return_rate=returned_rate.sort_values('returned_rate',ascending=False).head(1)
print(highest_return_rate)



                  total_orders  returned_orders  returned_rate
product_category                                              
Home                       306               30       0.098039


In [5]:
# Find customers who consistently rate products below the overall average rating.
overall_avg_rating=df['rating'].mean()
customer_avg=df.groupby('customer_id')['rating'].mean()
below_rating_customer=customer_avg[customer_avg < overall_avg_rating]
below_customer=below_rating_customer.reset_index(name='below_rating_customer')
print(below_customer)



     customer_id  below_rating_customer
0              1               2.333333
1              3               2.000000
2              4               1.500000
3              8               2.555556
4              9               2.750000
..           ...                    ...
158          290               2.500000
159          292               1.666667
160          295               2.400000
161          296               3.000000
162          299               3.000000

[163 rows x 2 columns]


In [6]:
# Identify cities where average product rating is below the global average rating.
global_avg=df['rating'].mean()
city_avg=df.groupby('city')['rating'].mean()
below_rating=city_avg[city_avg<global_avg]
low_rating_cities=below_rating.reset_index(name='low_rating_cities')
print(low_rating_cities)


        city  low_rating_cities
0  Bangalore           2.865672
1  Hyderabad           2.957447
2    Kolkata           2.816176
3     Mumbai           2.937063


In [7]:
# Calculate month-over-month growth rate in total revenue.
df['order_date']=pd.to_datetime(df['order_date'])
df['month']=df['order_date'].dt.to_period('M')
df['revenue']=df['price']*df['quantity']
month_revenue=df.groupby('month')['revenue'].sum()
mom_growth=month_revenue.pct_change()*100
print(mom_growth)


month
2023-01          NaN
2023-02    -0.324923
2023-03    10.406804
2023-04   -12.057660
2023-05   -22.552258
2023-06    29.648539
2023-07   -30.239552
2023-08    62.968608
2023-09   -22.919967
2023-10     8.377297
2023-11     6.079532
2023-12    -1.702592
2024-01    61.257735
2024-02   -36.815514
2024-03    26.405140
2024-04   -29.523826
2024-05    30.498830
2024-06   -17.146561
2024-07    23.912503
2024-08   -17.999607
2024-09    22.411802
2024-10    -9.054743
2024-11     5.590455
Freq: M, Name: revenue, dtype: float64


In [8]:
# Find customers whose spending increased every time they placed a new order.

